In [2]:
# copie o código total aqui

import random
import pyautogui
import pandas as pd
import pyperclip
import time
import subprocess
import numpy as np
'''
Retire a leitura do arquivo Excel, mas gere um dataframe sintético (tome como base o que já foi apresentado em aulas anteriores)

Adicionar uma coluna Categoria ao relatório final, classificando o pedido com base no valor do Total_Liquido,
da seguinte forma: Até RS 1.000: "Pequeno"; De RS 1.001 a RS 20.000: "Médio"; e Acima de RS 20.000: "Grande"

Altere a função exportar_relatorio() para que, além de salvar o arquivo, exiba no console um resumo
agrupado por status, mostrando a quantidade de pedidos e o total líquido de cada grupo. Dica: use groupby, len e sum.
'''
n = 11
PAUSA = 0.5

# ---- DICIONARIO COM PREÇO UNITARIO DE CADA PRODUTO ----
PRODUTO_VALOR_UNI = {
    'Notebook': 2500,
    'Monitor': 800,
    'Teclado': 150,
    'Mouse': 18,
    'Headset': 350,
    'Webcam': 220,
    'Cadeira Gamer': 1800,
    'Mesa Escritorio': 3200,
    'Impressoa': 950,
    'Nobreak': 600,
}

#Dicionario em lista de tuplas
itens = np.array(list(PRODUTO_VALOR_UNI.items()), dtype=object)

#Sorteando indices aleatorios
indices = np.random.randint(0, len(itens), n)


DATAFRAME_SINTETICO = pd.DataFrame({
    'ID_Pedido': range(1, n+1),
    'Produto': itens[indices][:, 0],
    'Quantidade': np.random.randint(1, 16, n),
    'Preco_Unitario': itens[indices][:, 1],
    'Desconto_%': np.random.randint(0, 16, n)
})

print(DATAFRAME_SINTETICO)

PLANILHA_PEDIDOS = DATAFRAME_SINTETICO
RELATORIO_SAIDA  = "relatorio_pedidos.xlsx"


# ─── ATIVIDADE 1: Ler pedidos.xlsx
def ler_planilha(caminho):
    df = pd.read_excel(caminho)
    df.columns = df.columns.str.strip()
    return df


# ─── ATIVIDADE 2: Abrir calculadora
def abrir_calculadora():
    subprocess.Popen("gnome-calculator")
    time.sleep(1.5)


# ─── ATIVIDADE 3: Fechar calculadora
def fechar_calculadora():
    pyautogui.hotkey("alt", "f4")
    time.sleep(0.5)


# ─── ATIVIDADE 4: Calcular total pela calculadora
def calcular_total(quantidade, preco_unit):
    abrir_calculadora()
    pyautogui.typewrite(str(quantidade), interval=0.05)
    pyautogui.press("*")
    pyautogui.typewrite(str(preco_unit), interval=0.05)
    pyautogui.press("=")
    time.sleep(PAUSA)
    pyautogui.hotkey("ctrl", "a")
    time.sleep(0.3)
    pyautogui.hotkey("ctrl", "c")
    time.sleep(0.3)
    fechar_calculadora()

    valor = pyperclip.paste()
    return float(valor.replace(",", "."))


# ─── ATIVIDADE 5: Aplicar desconto e classificar status
def processar_pedido(pedido_id, produto, qtd, preco, desconto_pct):
    total_bruto    = calcular_total(qtd, preco)
    valor_desconto = round(total_bruto * (desconto_pct / 100), 2)
    total_liquido  = round(total_bruto - valor_desconto, 2)
    status         = "Aprovado" if total_liquido <= 20000 else "Revisao"

# ---- CATEGORIA ----
    if total_liquido <= 1000:
        categoria = "Pequeno"
    elif total_liquido > 1000 and total_liquido <= 20000:
        categoria = "Medio"
    else:
        categoria = "Grande"

    print(f"Produto {pedido_id}: R$ {total_liquido:.2f} -> {status}")

    return {
        "ID_Pedido":     pedido_id,
        "Produto":       produto,
        "Quantidade":    qtd,
        "Preco_Unit":    preco,
        "Desconto_%":    desconto_pct,
        "Total_Bruto":   total_bruto,
        "Valor_Desc":    valor_desconto,
        "Total_Liquido": total_liquido,
        "Categoria":     categoria,
        "Status":        status
    }


'''
Altere a função exportar_relatorio() para que, além de salvar o arquivo, exiba no console um resumo
agrupado por status, mostrando a quantidade de pedidos e o total líquido de cada grupo. Dica: use groupby, len e sum.
'''

# ─── ATIVIDADE 6: Exportar relatório
def exportar_relatorio(resultados, caminho):
    df_out      = pd.DataFrame(resultados)
    total_geral = df_out["Total_Liquido"].sum()
    df_out.to_excel(caminho, index=False)
    print(f"\nTotal Geral dos Pedidos: R$ {total_geral:.2f}")
    print(f"Relatório salvo em: {caminho}")
# ---- RESUMO COM BASE NO STATUS ----
    print("\n[RESUMO - STATUS]")

    resumo = df_out.groupby("Status")
    
    for status, grupo in resumo:
        quantidade = len(grupo)
        total = grupo["Total_Liquido"].sum()

        print(f"{status}: {quantidade} pedidos | Total Líquido: R$ {total:.2f}")


# ─── MAIN: fluxo principal
def main():
    df = PLANILHA_PEDIDOS

    resultados = []

    for _, row in df.iterrows():
        resultado = processar_pedido(
            pedido_id   = row["ID_Pedido"],
            produto     = row["Produto"],
            qtd         = int(row["Quantidade"]),
            preco       = float(row["Preco_Unitario"]),
            desconto_pct= float(row.get("Desconto_%", 0))
        )
        resultados.append(resultado)

    exportar_relatorio(resultados, RELATORIO_SAIDA)


if __name__ == "__main__":
    main()


    ID_Pedido    Produto  Quantidade Preco_Unitario  Desconto_%
0           1    Monitor          13            800          11
1           2    Monitor           4            800           0
2           3    Monitor           1            800           7
3           4    Teclado          12            150          15
4           5    Monitor           8            800           3
5           6   Notebook           2           2500           8
6           7    Headset           9            350           5
7           8    Monitor           7            800          10
8           9  Impressoa           5            950           7
9          10      Mouse           8             18          14
10         11      Mouse           9             18           3



** (gnome-calculator:6610): WARNING **: 21:40:04.900: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:04.901: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:04.923: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:04.923: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:04.924: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:04.924: currency-provider.vala:386: Cannot use ECB rates as don't have EUR rate

** (gnome-calculator:6610): WARNING **: 21:40:05.832: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:05.883: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:06.553: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6610): WARNING **: 21:40:06

Produto 1: R$ 9256.00 -> Aprovado



** (gnome-calculator:6626): WARNING **: 21:40:09.082: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:09.083: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:09.104: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:09.105: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:09.105: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:09.106: currency-provider.vala:386: Cannot use ECB rates as don't have EUR rate

** (gnome-calculator:6626): WARNING **: 21:40:10.022: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:10.690: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:10.691: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6626): WARNING **: 21:40:10

Produto 2: R$ 3200.00 -> Aprovado


...

Produto 9: R$ 4417.50 -> Aprovado



** (gnome-calculator:6725): WARNING **: 21:40:42.288: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:42.289: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:42.310: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:42.311: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:42.311: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:42.312: currency-provider.vala:386: Cannot use ECB rates as don't have EUR rate

** (gnome-calculator:6725): WARNING **: 21:40:43.217: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:43.834: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:43.835: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6725): WARNING **: 21:40:43

Produto 10: R$ 123.84 -> Aprovado



** (gnome-calculator:6737): WARNING **: 21:40:46.389: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:46.390: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:46.412: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:46.413: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:46.413: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:46.413: currency-provider.vala:386: Cannot use ECB rates as don't have EUR rate

** (gnome-calculator:6737): WARNING **: 21:40:47.302: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:47.917: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:47.918: unit.vala:443: Failed to convert value: π*x/180

** (gnome-calculator:6737): WARNING **: 21:40:47

Produto 11: R$ 157.14 -> Aprovado

Total Geral dos Pedidos: R$ 38268.98
Relatório salvo em: relatorio_pedidos.xlsx

[RESUMO - STATUS]
Aprovado: 11 pedidos | Total Líquido: R$ 38268.98
